In [2]:
import pandas as pd

## Load The Dataset

In [5]:
sd = pd.read_csv('transactions.csv')
sub = pd.read_csv('subscriptions.csv')

In [6]:
sd = pd.read_csv("transactions.csv")
sub = pd.read_csv("subscriptions.csv")

print("Transactions Shape:", sd.shape)
print("Subscriptions Shape:", sub.shape)

Transactions Shape: (213341, 27)
Subscriptions Shape: (674, 12)


In [7]:
# Dataset Columns
print("Transactions Columns:")
print(sd.columns.tolist())

print("\nSubscriptions Columns:")
print(sub.columns.tolist())

# Missing Values
print("\nTransactions Missing Values:")
print(sd.isnull().sum()[sd.isnull().sum() > 0])

print("\nSubscriptions Missing Values:")
print(sub.isnull().sum()[sub.isnull().sum() > 0])

# Duplicate Rows
print("\nTransactions Duplicates:", sd.duplicated().sum())
print("Subscriptions Duplicates:", sub.duplicated().sum())

Transactions Columns:
['transaction_id', 'account_id', 'customer_id', 'timestamp', 'amount', 'currency', 'merchant_name', 'merchant_category', 'merchant_mcc', 'transaction_type', 'payment_channel', 'city', 'country', 'counterparty_iban', 'label', 'category', 'subcategory', 'is_subscription', 'is_salary', 'is_transfer', 'is_cash_withdrawal', 'balance_after_transaction', 'is_foreign', 'original_amount', 'original_currency', 'fx_rate', 'foreign_fee']

Subscriptions Columns:
['subscription_id', 'customer_id', 'account_id', 'merchant_name', 'category', 'subcategory', 'amount', 'frequency_months', 'billing_day', 'start_date', 'end_date', 'status']

Transactions Missing Values:
city                  99110
counterparty_iban    209343
dtype: int64

Subscriptions Missing Values:
end_date    564
dtype: int64

Transactions Duplicates: 0
Subscriptions Duplicates: 0


## Transactions Dataset Cleaning

In [8]:
sd.drop(
    columns=['city', 'counterparty_iban'],
    inplace=True
)

print("Dataset Shape:", sd.shape)
print("\nMissing Values:")
print(sd.isnull().sum().sum())

Dataset Shape: (213341, 25)

Missing Values:
0


## Subscriptions Dataset Check

In [9]:
print("Subscription Status:")
print(sub['status'].value_counts())

print("\nEnd Date Missing by Status:")
print(
    pd.crosstab(
        sub['status'],
        sub['end_date'].isnull(),
        margins=True
    )
)

Subscription Status:
status
active     572
churned    102
Name: count, dtype: int64

End Date Missing by Status:
end_date  False  True  All
status                    
active        8   564  572
churned     102     0  102
All         110   564  674


# Investigate 8 Active Subscriptions

In [10]:
active_end = sub[
    (sub['status'] == 'active') &
    (sub['end_date'].notna())
]

print(active_end[
    ['subscription_id', 'merchant_name',
     'start_date', 'end_date', 'status']
].to_string(index=False))

subscription_id    merchant_name start_date   end_date status
      S00000221 Childcare Center 2025-03-16 2026-10-14 active
      S00000329 Childcare Center 2024-04-30 2026-10-14 active
      S00000477 Childcare Center 2025-11-25 2026-10-14 active
      S00000553 Childcare Center 2024-04-25 2026-10-14 active
      S00000561 Childcare Center 2026-05-23 2026-10-14 active
      S00000601 Childcare Center 2024-12-02 2026-10-14 active
      S00000628 Childcare Center 2025-03-20 2026-10-14 active
      S00000642 Childcare Center 2026-08-26 2026-10-14 active


## Target Variable Analysis

In [11]:
print("Total Transactions:", len(sd))

print("\nSubscription Distribution:")
print(sd['is_subscription'].value_counts())

print("\nSubscription Percentage:")
print(
    sd['is_subscription'].value_counts(normalize=True) * 100
)

print("\nUnique Customers:")
print(sd['customer_id'].nunique())

print("\nUnique Merchants:")
print(sd['merchant_name'].nunique())

Total Transactions: 213341

Subscription Distribution:
is_subscription
False    205997
True       7344
Name: count, dtype: int64

Subscription Percentage:
is_subscription
False    96.557624
True      3.442376
Name: proportion, dtype: float64

Unique Customers:
100

Unique Merchants:
180


## Subscription Merchant Analysis

In [12]:
# Subscription Transactions
subs = sd[sd['is_subscription'] == True].copy()

# Top 15 Subscription Merchants
print("Top 15 Subscription Merchants:")

print(
    subs['merchant_name']
    .value_counts()
    .head(15)
)

# Customer-Merchant Recurring Groups
groups = subs.groupby(
    ['customer_id', 'merchant_name']
).size()

print("\nTotal Customer-Merchant Groups:", len(groups))

print("\nGroups with 1 Payment:", (groups == 1).sum())

print("Groups with 2+ Payments:", (groups >= 2).sum())

print("Groups with 3+ Payments:", (groups >= 3).sum())

print("\nMaximum Payments in One Group:", groups.max())

Top 15 Subscription Merchants:
merchant_name
ConEdison        643
Verizon          579
AT&T             576
Amazon Prime     517
National Grid    489
State Farm       485
Geico Auto       449
Xfinity          434
T-Mobile         429
Netflix          388
Disney+          364
Spotify          362
Blue Cross       358
Apple iCloud+    304
Hulu             207
Name: count, dtype: int64

Total Customer-Merchant Groups: 518

Groups with 1 Payment: 42
Groups with 2+ Payments: 476
Groups with 3+ Payments: 442

Maximum Payments in One Group: 36


## Recurring Payment Interval Analysis

In [13]:
# Convert timestamp
sd['timestamp'] = pd.to_datetime(sd['timestamp'])

# Select subscription transactions
subs = sd[sd['is_subscription'] == True].copy()

# Sort transactions
subs = subs.sort_values(
    ['customer_id', 'account_id', 'merchant_name', 'timestamp']
)

# Calculate payment intervals
subs['payment_interval'] = (
    subs.groupby(
        ['customer_id', 'account_id', 'merchant_name']
    )['timestamp']
    .diff()
    .dt.total_seconds() / 86400
)

# Remove missing intervals
intervals = subs['payment_interval'].dropna()

print("Total Valid Intervals:", len(intervals))

print("\nAverage Interval:", round(intervals.mean(), 2))
print("Median Interval:", round(intervals.median(), 2))

print("\nPayment Frequency Analysis:")

print("Weekly (6-8 days):",
      intervals.between(6, 8).sum())

print("Monthly (27-32 days):",
      intervals.between(27, 32).sum())

print("Quarterly (85-95 days):",
      intervals.between(85, 95).sum())

print("Yearly (350-380 days):",
      intervals.between(350, 380).sum())


Total Valid Intervals: 6825

Average Interval: 37.57
Median Interval: 30.81

Payment Frequency Analysis:
Weekly (6-8 days): 0
Monthly (27-32 days): 6100
Quarterly (85-95 days): 213
Yearly (350-380 days): 5


## Historical Features